# 06 - Detection evaluation on the frozen test split (YOLO11n vs YOLO11s)
Evaluates both trained detectors on the **test** split from notebook 03, which no earlier notebook has touched.

1. **Operating confidence is chosen on val** (the threshold with the highest F1), never on test. Test numbers are then reported at that threshold, so they are not tuned on test.
2. **Overall test metrics** for each model: precision, recall, mAP50, mAP50-95, per-class AP (one class, `car`), PR curve, F1-vs-confidence, P and R curves, and the confusion matrix at the operating confidence.
3. **Per-group test metrics**: per site, per camera, per climate, and flagged vs normal days (groups come from the split manifest). Small groups are flagged, not hidden.
4. A side-by-side comparison of the two models, saved as CSV/JSON and plots.

**Inputs (Add Input -> Your Work):** notebook 03 output (`yolo_f0.1/`, `artifacts/splits/`), notebook 04 output (`artifacts/train/yolo11n_f0.1/`), notebook 05 output (`artifacts/train/yolo11s_f0.1/`). If one model is missing the notebook evaluates the other and marks the run PARTIAL.
**Accelerator:** GPU T4 x2 (one GPU is used). Internet ON (installs `ultralytics==8.3.40`).

**How to run:** `SMOKE = True` + Run All (~3 min, 200 test and 200 val images) to check the pipeline. Then `SMOKE = False` + **Save Version -> Save & Run All**. Re-running skips a model whose record already exists with the same config hash.
Kaggle T4 timings in this notebook are cloud numbers only; edge latency is measured on the Orin.

In [ ]:
# --- Setup: get the project code onto Kaggle ---
# Preferred: attach the code as a Kaggle dataset (works for a private GitHub repo).
# Fallback: git clone (only works if the GitHub repo is public and Internet is ON).
import os, sys, glob, shutil, subprocess
REPO_URL = "https://github.com/Amith2277/Edge-Computing-EndSem.git"
REPO_DIR = "/kaggle/working/repo"
MARK = os.path.join("src", "common", "config.py")

def find_attached_code():
    for pat in ("/kaggle/input/*/", "/kaggle/input/*/*/", "/kaggle/input/*/*/*/"):
        for d in sorted(glob.glob(pat)):
            if os.path.isfile(os.path.join(d, MARK)):
                return d.rstrip("/")
    return None

if os.path.isdir(REPO_DIR):
    print("using existing", REPO_DIR)
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):  # cloned earlier: fetch the latest code
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, universal_newlines=True, env=env)
        print(r.stdout.strip())
else:
    attached = find_attached_code()
    if attached:
        shutil.copytree(attached, REPO_DIR)
        print("copied attached code dataset:", attached)
    else:
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, universal_newlines=True, env=env)
        print(r.stdout)
        if r.returncode != 0:
            raise SystemExit("No attached code dataset found and git clone failed (private repo?). "
                             "Upload adaptivepark-code.zip as a Kaggle dataset and Add Input it, then re-run.")
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("code at", REPO_DIR, "| files:", sorted(os.listdir(REPO_DIR))[:12])


In [ ]:
# Install the pinned ultralytics (same version as the Orin so the .pt loads there). No torch upgrade.
import importlib.metadata as md_
try:
    have = md_.version("ultralytics")
except md_.PackageNotFoundError:
    have = None
if have != "8.3.40":
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.3.40"],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)
    print(r.stdout[-2000:])
# torch >= 2.6 defaults torch.load(weights_only=True), which ultralytics 8.3.40 checkpoints do not satisfy.
os.environ.setdefault("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", "1")
# Training uses one GPU. Hiding the second one lets ultralytics use all 4 CPU cores for data loading
# (it divides the cores by the number of visible GPUs: 2 loader workers on "T4 x2"). Must be set before torch loads.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
import torch, ultralytics
# Turn off experiment-tracker integrations (Kaggle's ray breaks the raytune callback; wandb can wait for a login).
# Runs before any trainer exists. If you saw a raytune error earlier, restart the kernel first.
from src.training import yolo_train as yt
print("tracker integrations:", yt.disable_integrations())
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

In [ ]:
# Settings and inputs
SMOKE = True                      # True: 200 test + 200 val images. False: full test split (use Save & Run All)
VARIANTS = ["yolo11n", "yolo11s"]
DEVICE = 0
BATCH = 32
SMOKE_IMAGES = 200
PLOTS = True
INPUT_ROOT = "/kaggle/input"
WORK = "/kaggle/working"

import json, re, time, csv
from pathlib import Path
from src.common.config import load_yaml, config_hash
from src.common.status import new_run_id, write_status
from src.common.hashing import sha256_file
from src.training import yolo_train as yt
from src.evaluation import det_eval as de
DCFG = load_yaml("configs/dataset.yaml")
FRAC = DCFG["dataset"]["subset_fraction"]
IMGSZ = {v: load_yaml("configs/train_{}.yaml".format(v))["imgsz"] for v in VARIANTS}
EXPORT = yt.find_export_dir(INPUT_ROOT, FRAC)
mans = [p for p in yt.find_files(INPUT_ROOT, "split_manifest.csv") if p.parent.name == "splits"]
assert mans, "split_manifest.csv not found: attach notebook 03's output"
ROWS = de.read_manifest(mans[0])
SPLIT_INFO = json.loads((mans[0].parent / "split_info.json").read_text())
print("export:", EXPORT, "| manifest:", mans[0], "| hash:", SPLIT_INFO["manifest_hash"])

MODELS, MISSING = {}, []
records = yt.find_files(INPUT_ROOT, "training_record.json")
for v in VARIANTS:
    name = "{}_f{}".format(v, FRAC)
    found = [p for p in records if p.parent.name == name]
    if not found:
        MISSING.append(v); print("MISSING", name, "- attach the output of the notebook that trained it"); continue
    d = found[0].parent
    rec = json.loads(found[0].read_text())
    sha = sha256_file(d / "best.pt")
    assert sha == rec["checkpoint_sha256"], "{}: best.pt does not match its training record".format(v)
    assert not rec.get("smoke_test"), "{}: this is a smoke-test checkpoint".format(v)
    assert rec["dataset_split_hash"] == SPLIT_INFO["manifest_hash"], "{}: trained on a different split".format(v)
    MODELS[v] = {"weights": d / "best.pt", "sha256": sha, "train_record": rec}
    print("{}: {} | sha256 {}... | best epoch {} | val mAP50-95 {}".format(
        v, d, sha[:12], rec["best_epoch"]["epoch"], round(rec["val_metrics_best_pt"]["mAP50-95"], 4)))
assert MODELS, "no trained model attached"

# Copy val+test to local disk (the input mount is slow and read-only; ultralytics wants to write label caches)
LOCAL = Path(WORK) / "yolo_local_f{}".format(FRAC)
t0 = time.time(); print("staging:", yt.stage_export(EXPORT, LOCAL, splits=("val", "test")), "in", round(time.time() - t0, 1), "s")

In [ ]:
# Test groups from the split manifest (site, camera, climate, flagged/normal day)
test_names = sorted(p.name for p in (LOCAL / "images" / "test").iterdir())
val_names = sorted(p.name for p in (LOCAL / "images" / "val").iterdir())
if SMOKE:
    test_names, val_names = de.stride_sample(test_names, SMOKE_IMAGES), de.stride_sample(val_names, SMOKE_IMAGES)
GROUPS = de.test_groups(ROWS, test_names)
n_unmatched = len(test_names) - len(GROUPS["all"]["images"])
assert n_unmatched == 0, "{} test images are not in the manifest".format(n_unmatched)
GDIR = Path(WORK) / "eval_groups"
gid = lambda g: re.sub(r"[^A-Za-z0-9_.-]+", "_", g)
GROUP_YAML = {g: de.build_group_dataset(LOCAL, GDIR / gid(g), info["images"]) for g, info in GROUPS.items()}
VAL_YAML = de.build_group_dataset(LOCAL, GDIR / "_val", val_names, split="val")
for g, info in GROUPS.items():
    info["boxes"] = de.count_boxes(LOCAL / "labels" / "test", info["images"])
    info["note"] = de.small_group_note(len(info["images"]), info["days"])
print("{:<34} {:>7} {:>8} {:>5}  {}".format("group", "images", "cars", "days", "note"))
for g, info in GROUPS.items():
    print("{:<34} {:>7} {:>8} {:>5}  {}".format(g, len(info["images"]), info["boxes"], info["days"], info["note"]))
print("val images for choosing the confidence:", len(val_names))

In [ ]:
# Evaluate each model: confidence from val, then test overall + per group (restartable)
import numpy as np
from ultralytics import YOLO
OUT = Path(WORK) / "artifacts" / "eval_det"
OUT.mkdir(parents=True, exist_ok=True)
CFG_HASH = config_hash({"dataset": DCFG, "smoke": SMOKE, "smoke_images": SMOKE_IMAGES, "imgsz": IMGSZ, "batch": BATCH,
                        "weights": {v: m["sha256"] for v, m in MODELS.items()}, "split": SPLIT_INFO["manifest_hash"]})
RUN_ID = new_run_id("evaldet")
STATUS = OUT / "status.json"
write_status(STATUS, "06_eval_detection", RUN_ID, "RUNNING", CFG_HASH, "evaluating " + ", ".join(MODELS))
RESULTS = {}
try:
    for v, m in MODELS.items():
        vout = OUT / (v + ("_smoke" if SMOKE else ""))
        recp = vout / "eval_record.json"
        if recp.exists() and json.loads(recp.read_text()).get("config_hash") == CFG_HASH:
            RESULTS[v] = json.loads(recp.read_text()); print(v, ": record exists, skipped"); continue
        model = YOLO(str(m["weights"]))
        common = dict(imgsz=IMGSZ[v], batch=BATCH, device=DEVICE, verbose=False, exist_ok=True, project=str(vout))
        t_all = time.time()
        # 1) operating confidence: highest F1 on val (test is not used for any choice)
        vm = model.val(data=str(VAL_YAML), split="val", plots=False, name="val_choose_conf", **common)
        op = de.best_f1_conf(getattr(vm.box, "px", []), getattr(vm.box, "f1_curve", []))
        CONF = op["conf"] if isinstance(op["conf"], float) else None   # None: no detections on val (cannot choose)
        print("{}: operating confidence from val = {} (val F1 {})".format(v, CONF, op["f1"]))
        # 2) test, every group (the "all" group also saves ultralytics' PR/F1/P/R curves)
        groups = {}
        for g, y in GROUP_YAML.items():
            t0 = time.time()
            tm = model.val(data=str(y), split="test", plots=PLOTS and g == "all", name="test_" + gid(g), **common)
            s = de.box_summary(tm.box, CONF)
            info = GROUPS[g]
            s.update({"images": len(info["images"]), "cars": info["boxes"], "days": info["days"], "note": info["note"],
                      "eval_seconds": round(time.time() - t0, 1)})
            if g == "all":
                s["speed_ms_per_image_kaggle_t4"] = {k: round(x, 3) for k, x in tm.speed.items()}
                ap50, ap = list(tm.box.ap50), list(tm.box.ap)   # empty when the model made no detections
                s["per_class_ap"] = {"car": {"AP50": round(float(ap50[0]), 4) if ap50 else 0.0,
                                             "AP50-95": round(float(ap[0]), 4) if ap else 0.0}}
                if hasattr(tm.box, "px"):   # curves exist only if the model made detections
                    np.savez(str(vout / "test_curves.npz"), px=tm.box.px, prec_values=tm.box.prec_values,
                             f1=tm.box.f1_curve, p=tm.box.p_curve, r=tm.box.r_curve)
            groups[g] = s
            print("  {:<34} n={:<5} mAP50={:<7} mAP50-95={:<7} P/R/F1@conf={}".format(
                g, s["images"], s["mAP50"], s["mAP50-95"],
                [s.get("at_operating_conf", {}).get(k) for k in ("precision", "recall", "f1")]))
        # 3) confusion matrix at the operating confidence (a separate pass: passing conf changes mAP, so it is not used for mAP)
        cm = model.val(data=str(GROUP_YAML["all"]), split="test", conf=CONF or 0.25, plots=PLOTS,
                       name="test_all_at_operating_conf", **common)
        mat = cm.confusion_matrix.matrix
        cmd = {"conf_source": "val max-F1" if CONF else "ultralytics default 0.25 (no val detections)", "conf": CONF or 0.25, "iou": cm.confusion_matrix.iou_thres, "rows_pred_cols_true": ["car", "background"],
               "matrix": mat.tolist(), "true_positives": int(mat[0, 0]), "false_positives": int(mat[0, 1]),
               "false_negatives": int(mat[1, 0])}
        rec = {"run_id": RUN_ID, "config_hash": CFG_HASH, "smoke_test": SMOKE, "model_variant": v,
               "weights": str(m["weights"]), "checkpoint_sha256": m["sha256"], "dataset_split_hash": SPLIT_INFO["manifest_hash"],
               "split": "test", "imgsz": IMGSZ[v], "batch": BATCH, "environment": "Kaggle " + torch.cuda.get_device_name(0),
               "operating_conf": {"chosen_on": "val ({} images)".format(len(val_names)), "rule": "max F1 (class mean, unsmoothed)", **op},
               "test": groups["all"], "groups": {g: s for g, s in groups.items() if g != "all"},
               "confusion_matrix_at_operating_conf": cmd, "eval_seconds_total": round(time.time() - t_all, 1),
               "note": ("mAP uses ultralytics' default val settings (conf 0.001, IoU 0.7 NMS). precision/recall_at_best_f1 use the "
                        "threshold that is best on that same set (optimistic); at_operating_conf uses the val-chosen threshold.")}
        recp.write_text(json.dumps(rec, indent=2, default=str))
        RESULTS[v] = rec
        print(v, "done in {:.1f} min".format(rec["eval_seconds_total"] / 60), "->", recp)
except Exception as e:
    write_status(STATUS, "06_eval_detection", RUN_ID, "FAILED", CFG_HASH, repr(e)[:500])
    raise

In [ ]:
# Comparison table (test split) and per-group CSV
cols = ("images", "cars", "mAP50", "mAP50-95")
lines, csv_rows = [], []
for v, r in RESULTS.items():
    t = r["test"]; a = t.get("at_operating_conf", {})
    lines.append({"model": v, "operating_conf": r["operating_conf"]["conf"], **{c: t[c] for c in cols},
                  "precision": a.get("precision"), "recall": a.get("recall"), "f1": a.get("f1"),
                  "TP": r["confusion_matrix_at_operating_conf"]["true_positives"],
                  "FP": r["confusion_matrix_at_operating_conf"]["false_positives"],
                  "FN": r["confusion_matrix_at_operating_conf"]["false_negatives"],
                  "inference_ms_batched_t4": t["speed_ms_per_image_kaggle_t4"]["inference"],
                  "params": MODELS[v]["train_record"]["parameters"], "gflops": MODELS[v]["train_record"]["gflops_at_imgsz"]})
    for g, s in [("all", t)] + sorted(r["groups"].items()):
        a = s.get("at_operating_conf", {})
        csv_rows.append({"model": v, "group": g, "images": s["images"], "cars": s["cars"], "days": s["days"],
                         "mAP50": s["mAP50"], "mAP50-95": s["mAP50-95"], "precision": a.get("precision"),
                         "recall": a.get("recall"), "f1": a.get("f1"), "note": s["note"]})
for name, rows_ in (("comparison", lines), ("groups", csv_rows)):
    with open(str(OUT / "{}{}.csv".format(name, "_smoke" if SMOKE else "")), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows_[0].keys())); w.writeheader(); w.writerows(rows_)
print("TEST SPLIT (precision/recall/F1 at the val-chosen confidence)")
for l in lines:
    print(json.dumps(l))
print()
print("{:<8} {:<34} {:>6} {:>7} {:>9} {:>6} {:>6} {:>6}  {}".format("model", "group", "images", "mAP50", "mAP50-95", "P", "R", "F1", "note"))
for r_ in csv_rows:
    print("{:<8} {:<34} {:>6} {:>7} {:>9} {:>6} {:>6} {:>6}  {}".format(*[str(r_[k]) for k in (
          "model", "group", "images", "mAP50", "mAP50-95", "precision", "recall", "f1", "note")]))

In [ ]:
# Plots: PR and F1-vs-confidence (both models, test), and mAP50-95 per group
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for v, r in RESULTS.items():
    cp = OUT / (v + ("_smoke" if SMOKE else "")) / "test_curves.npz"
    if not cp.exists():
        print(v, ": no test curves (no detections)"); continue
    c = np.load(str(cp))
    ax[0].plot(c["px"], c["prec_values"].mean(0), label="{} (AP50 {})".format(v, r["test"]["mAP50"]))
    ax[1].plot(c["px"], c["f1"].mean(0), label=v)
    if isinstance(r["operating_conf"]["conf"], float):
        ax[1].axvline(r["operating_conf"]["conf"], ls="--", lw=0.8, color=ax[1].lines[-1].get_color())
ax[0].set(xlabel="recall", ylabel="precision", title="PR curve, test split (IoU 0.5)", xlim=(0, 1), ylim=(0, 1.02))
ax[1].set(xlabel="confidence", ylabel="F1", title="F1 vs confidence, test (dashed: conf chosen on val)", xlim=(0, 1), ylim=(0, 1.02))
for a in ax: a.legend(); a.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(str(OUT / "pr_f1_test{}.png".format("_smoke" if SMOKE else "")), dpi=120); plt.show()

kinds = ["site", "climate", "day_type", "camera"]
fig, ax = plt.subplots(len(kinds), 1, figsize=(13, 3.2 * len(kinds)))
for a, k in zip(ax, kinds):
    gs = sorted({g for r in RESULTS.values() for g in r["groups"] if g.startswith(k + ":")})
    x = np.arange(len(gs)); w = 0.8 / max(1, len(RESULTS))
    for i, (v, r) in enumerate(RESULTS.items()):
        vals = [r["groups"][g]["mAP50-95"] if g in r["groups"] else np.nan for g in gs]
        a.bar(x + i * w, vals, w, label=v)
    a.set_xticks(x + w * (len(RESULTS) - 1) / 2)
    a.set_xticklabels(["{}\n(n={}){}".format(g.split(":", 1)[1].replace("CNRPark-EXT/", "CNR/"), len(GROUPS[g]["images"]),
                                              " *" if GROUPS[g]["note"] else "") for g in gs], fontsize=8)
    a.set_ylim(0, 1); a.set_ylabel("mAP50-95"); a.set_title("test mAP50-95 by " + k + "  (* = small group)"); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig(str(OUT / "groups_test{}.png".format("_smoke" if SMOKE else "")), dpi=110); plt.show()

In [ ]:
# Ultralytics' own test plots for each model (PR, F1, confusion matrix at the operating confidence)
from IPython.display import Image as IPImage, display
for v in RESULTS:
    vout = OUT / (v + ("_smoke" if SMOKE else ""))
    for sub, f in (("test_all", "PR_curve.png"), ("test_all", "F1_curve.png"),
                   ("test_all_at_operating_conf", "confusion_matrix.png")):
        p = vout / sub / f
        if p.exists():
            print(v, sub, f); display(IPImage(filename=str(p), width=520))
        else:
            print(v, "not found:", p)

In [ ]:
# Finish: environment, status, clean up local copies in committed runs
from src.common.env_record import save_environment
save_environment(str(OUT / "environment.json"))
state = "COMPLETE" if not MISSING and len(RESULTS) == len(VARIANTS) else "PARTIAL"
write_status(STATUS, "06_eval_detection", RUN_ID, state, CFG_HASH,
             "smoke={} evaluated={} missing={}".format(SMOKE, sorted(RESULTS), MISSING),
             extra={v: {"test_mAP50-95": r["test"]["mAP50-95"], "operating_conf": r["operating_conf"]["conf"]} for v, r in RESULTS.items()})
print(state, "| saved to", OUT, sorted(p.name for p in OUT.iterdir()))
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch":
    for d in (LOCAL, GDIR):
        shutil.rmtree(str(d), ignore_errors=True)
    print("removed staged copies")

## What to send back
Paste the output of the **Settings**, **groups**, **evaluate** and **comparison** cells, plus the two plots (PR/F1 and per-group).
After the full run (`SMOKE = False`, Save & Run All), the version's output holds `artifacts/eval_det/` (records, CSVs, plots). Notebook 07 (occupancy) attaches it together with notebooks 03-05.